In [ ]:
import rebound, numpy as np, time
from rebound import clibrebound

# function for resolving collission with merging if they touch
def probabilistic_sticking_resolve(sim, collision):

    sim_contents = sim.contents
    p1 = sim_contents.particles[collision.p1]
    p2 = sim_contents.particles[collision.p2]

    key = frozenset((p1.name, p2.name))
    step = sim_contents.steps_done

    prev = contacts.get(key)
    if (prev is None or prev[0] < step - 1):
        stick = rng.uniform(0,1) <= prob_stick
    else:
        stick = prev[1]
    contacts[key] = (step, stick)

    clibrebound = rebound.clibrebound
    if stick:
        del contacts[key]
        return clibrebound.reb_collision_resolve_merge(sim, collision)
    return clibrebound.reb_collision_resolve_hardsphere(sim, collision)

# hit a wall? reflect your velocity
def reflect(sim):
    N = sim.N
    xyz, v, r = np.empty((N, 3)), np.empty((N, 3)), np.empty(N)
    sim.serialize_particle_data(xyz=xyz, vxvyvz=v, r=r)
    r = r[:, None]

    hi = (xyz + r > half) & (v > 0)
    lo = (xyz - r < -half) & (v < 0)
    if not (hi.any() or lo.any()):
        return                      

    xyz = np.where(hi, half - r, np.where(lo, -half + r, xyz))
    v = np.where(hi | lo, -v, v)
    sim.set_serialized_particle_data(xyz=xyz, vxvyvz=v)

# from inches of the box to meters!
inch = 0.0254                                   
# half-widths per axis, m
half = np.array([0.8, 0.8, 1.55]) * inch / 2    

# Graphite: 5 mg of 125 um diameter spheres, density 2.26 g/cm^3

# particle radius in meters
r_p = 62.5e-6               

 # mass = 2.26 g/cm^3 * (4/3) pi r_p^3 in grams
m_p = 2.311e-6                                

# 5 mg / m_p, rounded down. 5 mg is arbitrary
N_p = 2163                                      

# the velocity spread for computing a good dt and for the normal dist!!
v_sigma = 0.01                                  

# initialization
sim = rebound.Simulation()
sim.units = ('m', 's', 'g')
sim.start_server(port=8000)
sim.integrator = "leapfrog"
sim.gravity = "none"                  

 # low enough dt so there's a very low probability for a particle to be faster than the dt!
sim.dt = 0.1 * r_p / (16 * v_sigma)        

# REBOUND's box is a cube; must enclose the tallest side
sim.root_size = 2 * half.max()                   
sim.boundary = "open" 

# best collission method for speed
sim.collision = "linetree"                       
sim.collision_resolve = probabilistic_sticking_resolve

# 1 for testing purposes
prob_stick = 1                                   
rng = np.random.default_rng(42)
contacts = {}

# add the particles
for i in range(N_p):
    sim.add(m=m_p, r=r_p,
            x=rng.uniform(-half[0] + r_p, half[0] - r_p),
            y=rng.uniform(-half[1] + r_p, half[1] - r_p),
            z=rng.uniform(-half[2] + r_p, half[2] - r_p),
            vx=rng.normal(0, v_sigma), vy=rng.normal(0, v_sigma), vz=rng.normal(0, v_sigma),
            name = f"p{i}")

t_end = 5.0
frames = []
save_every = 30
for k, t in enumerate(np.arange(0, t_end, sim.dt)):

    sim.integrate(t)
    reflect(sim)

    if k % save_every == 0:
        N = sim.N
        xyz, r = np.empty((N, 3)), np.empty(N)
        sim.serialize_particle_data(xyz=xyz, r=r)
        frames.append((sim.t, xyz, r))
    # time.sleep(0.01)

REBOUND Webserver listening on http://localhost:8000 (not secure) ...


In [ ]:
import pyvista as pv
pv.global_theme.notebook = False     # open a real window, not inline in the notebook

def spheres(xyz, r):

    # think of cloud as storage of our particle measurements, including position and radius!
    cloud = pv.PolyData(xyz)
    cloud["r"] = r

    # put a sphere at each point, scaled by its radius
    return cloud.glyph(scale="r", orient=False, factor=1.0,
                       geom=pv.Sphere(radius=1.0, theta_resolution=12, phi_resolution=12))
    # theta and phi are just resolution of each particle. lower if video is loading too slow

# video quality
pl = pv.Plotter(window_size=(1600, 1600))

# where to draw box edges wrt to halves
pl.add_mesh(pv.Box(bounds=(-half[0], half[0], -half[1], half[1], -half[2], half[2])), style="wireframe", color="gray")

# add the spheres
xyz, r = frames[0][1:]
actor = pl.add_mesh(spheres(xyz, r), color="orange", smooth_shading=True)

# text for time and number of particles in the simulation. the collissions will result in fewer particles because of the merging process.
label = pl.add_text("", font_size=12)

# opens the window, returns right away
pl.show(interactive_update=True, auto_close=False)
pl.open_movie("collisions.mp4", framerate=30, quality=6)

# replay the frames saved
for t, xyz, r in frames:                          
    actor.mapper.SetInputData(spheres(xyz, r))
    label.SetText(2, f"t = {t:.3f} s   N = {len(r)}")
    pl.update()
    pl.write_frame()
pl.close() 